### Importação das Bibliotecas

In [63]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

### Carregando Dados

In [64]:
df_vendas   = pd.read_csv('../data/processed/vendas_2023_2024_processed.csv')
df_produtos = pd.read_csv('../data/processed/produtos_processed.csv')

In [65]:
df_vendas[['id_client', 'id_product', 'qtd']].head()

,id_client,id_product,qtd
0,42,105,11
1,3,136,9
2,25,139,7
3,20,23,5
4,8,57,4


### Matriz Usuário × Produto (presença/ausência)

In [66]:
df_interacao = df_vendas.groupby(['id_client', 'id_product'])['qtd'].sum().reset_index()

df_interacao['comprou'] = 1

# Pivota: linhas = clientes, colunas = produtos, valor = 0 ou 1
matriz = df_interacao.pivot_table(
    index='id_client',
    columns='id_product',
    values='comprou',
    fill_value=0
)

matriz.shape

(49, 150)

In [67]:
matriz.iloc[:5, :5]

id_product,1,2,3,4,5
id_client,,,,,
1,1.0,0.0,1.0,1.0,1.0
2,0.0,1.0,1.0,0.0,1.0
3,1.0,1.0,1.0,1.0,1.0
4,1.0,1.0,0.0,1.0,1.0
5,1.0,1.0,0.0,1.0,1.0


### Similaridade de Cosseno entre produtos

In [68]:
matriz_T = matriz.T

sim = cosine_similarity(matriz_T)

df_similares = pd.DataFrame(
    sim,
    index=matriz_T.index,
    columns=matriz_T.index
)

df_similares.shape

(150, 150)

In [69]:
df_similares.iloc[:5, :5].round(2)


id_product,1,2,3,4,5
id_product,,,,,
1,1.00,0.78,0.74,0.81,0.75
2,0.78,1.00,0.70,0.76,0.71
3,0.74,0.70,1.00,0.80,0.70
4,0.81,0.76,0.80,1.00,0.76
5,0.75,0.71,0.70,0.76,1.00


### Busca por ID

In [70]:
produto_referencia = 'GPS Garmin Vortex Maré Drift'
id_ref = df_produtos[df_produtos['name'] == produto_referencia]['code'].values[0]
print(f"\nProduto de referência: {produto_referencia} (ID: {id_ref})")


Produto de referência: GPS Garmin Vortex Maré Drift (ID: 27)


### Rank dos 5 mais similares

In [71]:
similares = df_similares[id_ref].drop(id_ref)  
similares

id_product
1      0.850000
2      0.748331
3      0.764217
4      0.784873
5      0.694879
         ...   
146    0.721605
147    0.795133
148    0.775058
149    0.805807
150    0.775058
Name: 27, Length: 149, dtype: float64

In [72]:

top5 = similares.sort_values(ascending=False).head(5).reset_index()
top5.columns = ['id_product', 'similaridade']
top5 = top5.merge(df_produtos[['code', 'name']], left_on='id_product', right_on='code')

print("Top 3 produtos recomendados")
print(top5[['code', 'name', 'similaridade']].to_string(index=False))


Top 3 produtos recomendados
 code                                       name  similaridade
   94           Motor de Popa Volvo Magnum 276HP      0.869626
   11        GPS Furuno Swift Leviathan Poseidon      0.868037
   35                         Radar Furuno Swift      0.853913
    1                Transponder AIS Maré Magnum      0.850000
  115 Cabo de Nylon Delta Force Magnum Leviathan      0.850000
